# ByT5 Fine-Tuning: Tamil OCR Error Correction
> **Model:** `google/byt5-small`  
> **Data:** Loaded directly from Hugging Face Dataset Hub  
> **Output:** Trained model pushed directly to Hugging Face Model Hub  
> **Resumption:** Checkpoints pushed to HF Hub after every epoch — restart-safe

## 1. Install Dependencies

In [ ]:
%%capture
!pip install -U accelerate peft transformers
!pip install -q datasets==2.19.1 evaluate==0.4.2 sentencepiece huggingface_hub jiwer

## 2. Authenticate with Hugging Face
> Add your HF token to **Kaggle Secrets** (Add-ons → Secrets) under the name `HF_TOKEN`.

In [1]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

hf_token = UserSecretsClient().get_secret('HF_TOKEN')
login(token=hf_token, add_to_git_credential=False)
print('Logged in to Hugging Face.')

Logged in to Hugging Face.


## 3. Config

In [2]:
import os, re, unicodedata
import torch
import numpy as np
from pathlib import Path

# ── Identity ──────────────────────────────────────────────────────────────────
HF_USERNAME     = 'Naren-hug'                   # <── change this
DATASET_REPO    = f'{HF_USERNAME}/tamil-ocr-byt5-dataset'
MODEL_HUB_REPO  = f'{HF_USERNAME}/byt5-tamil-ocr-v1'  # where model is saved

# ── Model ─────────────────────────────────────────────────────────────────────
BASE_MODEL      = 'google/byt5-small'

# ── Paths ─────────────────────────────────────────────────────────────────────
OUTPUT_DIR      = Path('/kaggle/working/byt5-tamil-ocr')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Hyper-params ──────────────────────────────────────────────────────────────
MAX_INPUT_LEN   = 1024
MAX_TARGET_LEN  = 1024
TRAIN_BATCH     = 2       # per-device; safe for T4 16GB with fp16 + grad-ckpt
GRAD_ACCUM      = 8       # effective batch = 16
LR              = 5e-4
EPOCHS          = 5
WARMUP_STEPS    = 200

print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), 'GB')

GPU: Tesla T4
VRAM: 15.6 GB


## 4. Load Dataset from Hugging Face

In [3]:
from datasets import load_dataset

HTML_RE = re.compile(r'<[^>]+'+'>')

def clean(batch):
    """Strip residual HTML tags and NFC-normalise Unicode."""
    batch['input']  = [unicodedata.normalize('NFC', HTML_RE.sub('', t)).strip() for t in batch['input']]
    batch['target'] = [unicodedata.normalize('NFC', HTML_RE.sub('', t)).strip() for t in batch['target']]
    return batch

raw = load_dataset(DATASET_REPO, token=hf_token)
raw = raw.map(clean, batched=True)

# Drop identical input==target pairs (Surya read the image perfectly — useless for training)
raw = raw.filter(lambda x: x['input'] != x['target'])

print(raw)
print('Sample:', raw['train'][0])

Generating train split:   0%|          | 0/9001 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/9001 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/9001 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input', 'target', 'source'],
        num_rows: 9001
    })
    validation: Dataset({
        features: ['input', 'target', 'source'],
        num_rows: 1000
    })
})
Sample: {'input': 'ு உடு அவன் ஆங்கு எனக்கு அருள்ளெடும் உரபைபளேன் "சம்புத் தீவினுள் தமிழக மருங்கில் கம்பம் இல்லாக கழி பரெளு சுவலைய', 'target': 'ஆங்கு அவன் ஆங்கு எனக்கு அருளொடும் உரைப்போன் "சம்புத் தீவினுள் தமிழக மருங்கில் கம்பம் இல்லாக் கழி பெருஞ் செல்வர்', 'source': 'roundtrip_surya'}


## 5. Tokenise

In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def tokenize(batch):
    model_inputs = tokenizer(
        batch['input'],
        max_length=MAX_INPUT_LEN,
        truncation=True,
        padding=False,
    )
    labels = tokenizer(
        text_target=batch['target'],
        max_length=MAX_TARGET_LEN,
        truncation=True,
        padding=False,
    )
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

remove_cols = [c for c in raw['train'].column_names if c not in ('input_ids', 'attention_mask', 'labels')]
tok = raw.map(tokenize, batched=True, remove_columns=remove_cols)

print('Tokenisation done.')
print('Max input len in train:', max(len(x) for x in tok['train']['input_ids']))

config.json:   0%|          | 0.00/698 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/9001 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenisation done.
Max input len in train: 965


## 6. Load Model

In [5]:
from transformers import AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)
model.config.use_cache = False  # required when gradient_checkpointing=True

print(f'Parameters: {sum(p.numel() for p in model.parameters())/1e6:.1f}M')

pytorch_model.bin:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/172 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Parameters: 299.6M


## 7. Metric: Character Error Rate (CER)

In [6]:
import evaluate

cer_metric = evaluate.load('cer')

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    if isinstance(preds, tuple):
        preds = preds[0]
        
    # Replace -100 in labels as we can't decode them
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    
    # NEW: Sanitize predictions (ByT5 vocab is 0-384)
    # This prevents the "chr() arg not in range" crash
    preds = np.where((preds >= 0) & (preds < tokenizer.vocab_size), preds, tokenizer.pad_token_id)

    decoded_preds  = [p.strip() for p in tokenizer.batch_decode(preds,  skip_special_tokens=True)]
    decoded_labels = [l.strip() for l in tokenizer.batch_decode(labels, skip_special_tokens=True)]
    
    # Compute Character Error Rate
    result = cer_metric.compute(predictions=decoded_preds, references=decoded_labels)
    return {'cer': round(result, 4)}

## 8. Detect Existing Checkpoint (for Resumption)

In [7]:
import glob

def get_latest_checkpoint(output_dir):
    """Scan for the latest checkpoint-NNNN directory."""
    checkpoints = sorted(
        glob.glob(str(output_dir / 'checkpoint-*')),
        key=lambda p: int(p.split('-')[-1])
    )
    if checkpoints:
        print(f'Found checkpoint: {checkpoints[-1]}')
        return checkpoints[-1]
    print('No local checkpoint found — starting from scratch.')
    return None

resume_from = get_latest_checkpoint(OUTPUT_DIR)

No local checkpoint found — starting from scratch.


## 9. Training Arguments

In [8]:
from transformers import Seq2SeqTrainingArguments, DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,   # Use 'tokenizer' here
    model=model,
    label_pad_token_id=-100,
    pad_to_multiple_of=8,
)

training_args = Seq2SeqTrainingArguments(
    output_dir=str(OUTPUT_DIR),

    # ── Batching ─────────────────────────────────────────────────────────────
    per_device_train_batch_size=TRAIN_BATCH,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=GRAD_ACCUM,

    # ── Memory ───────────────────────────────────────────────────────────────
    gradient_checkpointing=True,   # halves activation memory on T4
    fp16=True,                     # T4 supports fp16, NOT bf16

    # ── Optimizer & Scheduler ────────────────────────────────────────────────
    optim='adafactor',
    learning_rate=LR,
    lr_scheduler_type='cosine',
    warmup_steps=WARMUP_STEPS,

    # ── Epochs ───────────────────────────────────────────────────────────────
    num_train_epochs=EPOCHS,

    # ── Evaluation & Checkpointing ───────────────────────────────────────────
    eval_strategy='epoch',
    save_strategy='epoch',          # save a local checkpoint each epoch
    load_best_model_at_end=True,
    metric_for_best_model='cer',
    greater_is_better=False,
    save_total_limit=2,             # keep only 2 local checkpoints to save disk

    # ── Hugging Face Hub Push ─────────────────────────────────────────────────
    push_to_hub=True,
    hub_model_id=MODEL_HUB_REPO,
    hub_strategy='checkpoint',      # push to HF Hub after EVERY epoch checkpoint
                                    # ensures resumption even if Kaggle VM dies
    hub_token=hf_token,

    # ── Generation ───────────────────────────────────────────────────────────
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LEN,

    # ── Logging ──────────────────────────────────────────────────────────────
    logging_steps=50,
    report_to='none',
)

## 10. Train (auto-resumes if checkpoint exists)

In [9]:
from transformers import Seq2SeqTrainer, EarlyStoppingCallback

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tok['train'],
    eval_dataset=tok['validation'],
    processing_class=tokenizer,  # Keep 'processing_class' here
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

# resume_from is None on first run (trains from scratch)
# resume_from is a checkpoint path if this cell is re-run (resumes automatically)
trainer.train(resume_from_checkpoint=resume_from)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Cer
1,1.714108,0.087478,0.170300
2,1.251764,0.070348,0.127300
3,1.064159,0.063657,0.104200
4,0.934304,0.058747,0.098400
5,0.820113,0.059387,0.098700


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


TrainOutput(global_step=1410, training_loss=2.154495078282999, metrics={'train_runtime': 20240.6794, 'train_samples_per_second': 2.223, 'train_steps_per_second': 0.07, 'total_flos': 4.362332094133862e+16, 'train_loss': 2.154495078282999, 'epoch': 5.0})

## 11. Push Final Best Model to Hub

In [10]:
trainer.push_to_hub(commit_message='Final best model after training')
print(f'Model live at: https://huggingface.co/{MODEL_HUB_REPO}') 

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Model live at: https://huggingface.co/Naren-hug/byt5-tamil-ocr-v1


## 12. Quick Inference Test

In [12]:
# --- Manual Inference Test ---
test_text = "தமழ் ஒசிஆர் ததாழில்நந்பம்"  # Example with errors

inputs = tokenizer(test_text, return_tensors="pt", truncation=True).to(model.device)

with torch.no_grad():
    outputs = model.generate(inputs["input_ids"], max_length=128, num_beams=4)

corrected = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(f"RAW OCR:   {test_text}")
print(f"BYT5 FIX:  {corrected}")


RAW OCR:   தமழ் ஒசிஆர் ததாழில்நந்பம்
BYT5 FIX:  தமழ் ஒசிஆர் தாழில்நந்பம்
